# 02 — Unity Catalog registration, serving handoff, and feedback

This notebook logs `02_agent_graph.py` with the MLflow LangChain flavor, adds a version to the pre-provisioned `main.ai_platform.deepagent_supervisor` registered model, renders an executable platform-owner Model Serving handoff, validates the deployed version, invokes it, and attaches governed feedback to MLflow trace IDs.

Repository policy reserves endpoint and registered-model creation for an approved platform process. The notebook never creates or mutates an endpoint. It fails clearly until the external handoff has deployed the exact version. `InMemorySaver` remains available only for a one-worker demonstration; configure a platform-packaged durable checkpointer factory for production HITL.



In [ ]:
# ruff: noqa: E501, F404, F821

In [ ]:
%pip install "deepagents==0.7.5" "mlflow[databricks,langchain]==3.15.1" "langchain==1.3.14" "langgraph==1.2.9" "databricks-langchain==0.20.0" "databricks-sdk==0.122.0"

In [ ]:
%restart_python

In [ ]:
from __future__ import annotations

import json
import os
import py_compile
from enum import StrEnum
from pathlib import Path
from typing import Any
from uuid import UUID, uuid4

import mlflow
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (
    EndpointCoreConfigInput,
    EndpointStateConfigUpdate,
    EndpointStateReady,
    EndpointTag,
    Route,
    ServedEntityInput,
    TrafficConfig,
)
from mlflow import MlflowClient
from mlflow.entities import AssessmentSource, AssessmentSourceType
from mlflow.exceptions import MlflowException
from mlflow.models import infer_signature
from mlflow.models.resources import (
    DatabricksServingEndpoint,
    DatabricksSQLWarehouse,
    DatabricksVectorSearchIndex,
)
from pydantic import BaseModel, ConfigDict, Field, field_validator


def define_widget(name: str, default: str, label: str) -> str:
    dbutils.widgets.text(name, default, label)
    return dbutils.widgets.get(name).strip()


class DeploymentConfig(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, str_strip_whitespace=True)

    experiment_name: str = Field(pattern=r"^/.+")
    uc_model_name: str
    agent_module: Path
    model_endpoint: str = Field(pattern=r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,127}$")
    sql_warehouse_id: str = Field(pattern=r"^[A-Za-z0-9][A-Za-z0-9_-]{0,127}$")
    docs_index: str
    skill_uri: str
    serving_endpoint: str = Field(pattern=r"^[A-Za-z0-9][A-Za-z0-9_.-]{0,127}$")
    checkpointer_factory: str = ""
    application: str = "deepagents-solution-accelerator"
    project: str = "deepagents-supervisor"
    environment: str = Field(pattern=r"^[A-Za-z0-9_.:-]{1,80}$")
    data_classification: str = Field(pattern=r"^[A-Za-z0-9_.:-]{1,80}$")
    lifecycle: str = "validation"
    tag_schema_version: str = "2"

    @field_validator("uc_model_name")
    @classmethod
    def validate_uc_name(cls, value: str) -> str:
        parts = value.split(".")
        if len(parts) != 3 or any(
            not part.replace("_", "").replace("-", "").isalnum() for part in parts
        ):
            raise ValueError("uc_model_name must be catalog.schema.model")
        return value

    @field_validator(
        "model_endpoint",
        "sql_warehouse_id",
        "docs_index",
    )
    @classmethod
    def reject_placeholders(cls, value: str) -> str:
        if value.startswith("REPLACE_"):
            raise ValueError("replace all resource and governance placeholders")
        return value

    @field_validator("docs_index")
    @classmethod
    def validate_docs_index(cls, value: str) -> str:
        parts = value.split(".")
        if len(parts) != 3 or any(
            not part.replace("_", "").replace("-", "").isalnum() for part in parts
        ):
            raise ValueError("docs_index must be catalog.schema.index")
        return value

    @field_validator("skill_uri")
    @classmethod
    def validate_skill_uri(cls, value: str) -> str:
        allowed = value.startswith(
            "dbfs:/FileStore/deepagents/skills/"
        ) or value.startswith("/Volumes/")
        normalized = value.removeprefix("dbfs:")
        parts = normalized.split("/")[1:]
        volume_shape_ok = not value.startswith("/Volumes/") or len(parts) >= 6
        if (
            not allowed
            or not volume_shape_ok
            or any(part in {"", ".", ".."} for part in parts)
            or not value.endswith("/sql-governance/SKILL.md")
        ):
            raise ValueError(
                "skill_uri must target the approved sql-governance/SKILL.md path"
            )
        return value


config = DeploymentConfig(
    experiment_name=define_widget(
        "experiment_name",
        "/Shared/deepagents-solution-accelerator",
        "MLflow experiment",
    ),
    uc_model_name=define_widget(
        "uc_model_name", "main.ai_platform.deepagent_supervisor", "Unity Catalog model"
    ),
    agent_module=Path(
        define_widget("agent_module", "02_agent_graph.py", "Models-from-Code module")
    ),
    model_endpoint=define_widget(
        "model_endpoint", "REPLACE_MODEL_ENDPOINT", "Chat model endpoint"
    ),
    sql_warehouse_id=define_widget(
        "sql_warehouse_id", "REPLACE_SQL_WAREHOUSE", "SQL warehouse ID"
    ),
    docs_index=define_widget(
        "docs_index", "REPLACE_CATALOG.SCHEMA.DOCS_INDEX", "Documentation index"
    ),
    skill_uri=define_widget(
        "skill_uri",
        "dbfs:/FileStore/deepagents/skills/sql-governance/SKILL.md",
        "Remote SKILL.md path",
    ),
    serving_endpoint=define_widget(
        "serving_endpoint", "deepagent-supervisor", "Pre-provisioned serving endpoint"
    ),
    checkpointer_factory=define_widget(
        "checkpointer_factory", "", "Durable checkpointer module.path:factory"
    ),
    environment=define_widget("environment", "dev", "Environment tag"),
    data_classification=define_widget(
        "data_classification", "internal", "Data classification tag"
    ),
)

os.environ.update(
    {
        "DEEPAGENTS_MODEL_ENDPOINT": config.model_endpoint,
        "DEEPAGENTS_SQL_WAREHOUSE_ID": config.sql_warehouse_id,
        "DEEPAGENTS_DOCS_INDEX": config.docs_index,
        "DEEPAGENTS_SKILL_URI": config.skill_uri,
        "DEEPAGENTS_REQUIRE_REMOTE_SKILLS": "true",
        "DEEPAGENTS_APPLICATION": config.application,
        "DEEPAGENTS_ENVIRONMENT": config.environment,
    }
)
mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")
active_experiment = mlflow.set_experiment(config.experiment_name)
EXPERIMENT_ID = str(active_experiment.experiment_id)
if config.checkpointer_factory:
    os.environ["DEEPAGENTS_CHECKPOINTER_FACTORY"] = config.checkpointer_factory

agent_module = config.agent_module.expanduser().resolve()
if not agent_module.is_file():
    raise FileNotFoundError(f"Run notebook 01 first; module not found: {agent_module}")
py_compile.compile(str(agent_module), doraise=True)

## Log and register the Models-from-Code artifact

Logging passes the source path rather than a live graph, preventing cross-environment serialization failures. The signature is explicit, and every runtime dependency is pinned. Declared MLflow resources allow Databricks to infer serving dependencies without embedding credentials.



In [ ]:
INPUT_EXAMPLE = {
    "thread_id": "00000000-0000-4000-8000-000000000001",
    "messages": [
        {"role": "user", "content": "Summarize monthly revenue without executing SQL."}
    ],
    "decisions": [],
}
OUTPUT_EXAMPLE = {
    "trace_id": "tr-signature-example",
    "thread_id": "00000000-0000-4000-8000-000000000001",
    "status": "completed",
    "messages": [{"role": "assistant", "content": "Example response."}],
    "interrupts": [],
}
MODEL_SIGNATURE = infer_signature(INPUT_EXAMPLE, OUTPUT_EXAMPLE)
PIP_REQUIREMENTS = [
    "deepagents==0.7.5",
    "mlflow[databricks,langchain]==3.15.1",
    "langchain==1.3.14",
    "langgraph==1.2.9",
    "databricks-langchain==0.20.0",
    "databricks-sdk==0.122.0",
]

registry = MlflowClient(registry_uri="databricks-uc")
try:
    registry.get_registered_model(config.uc_model_name)
except MlflowException as exc:
    if getattr(exc, "error_code", None) == "RESOURCE_DOES_NOT_EXIST":
        raise RuntimeError(
            "Unity Catalog model is not pre-provisioned; request it through the platform process"
        ) from exc
    raise

with mlflow.start_run(run_name="register-deepagent-supervisor") as active_run:
    model_info = mlflow.langchain.log_model(
        lc_model=str(agent_module),
        name="deepagent_supervisor",
        registered_model_name=config.uc_model_name,
        input_example=INPUT_EXAMPLE,
        signature=MODEL_SIGNATURE,
        pip_requirements=PIP_REQUIREMENTS,
        resources=[
            DatabricksServingEndpoint(endpoint_name=config.model_endpoint),
            DatabricksVectorSearchIndex(index_name=config.docs_index),
            DatabricksSQLWarehouse(warehouse_id=config.sql_warehouse_id),
        ],
        model_type="agent",
        metadata={
            "application": config.application,
            "compatibility_path": "models-from-code-model-serving",
        },
    )

matching_versions = [
    version
    for version in registry.search_model_versions(f"name='{config.uc_model_name}'")
    if version.run_id == active_run.info.run_id
]
if len(matching_versions) != 1:
    raise RuntimeError(
        "Could not resolve the uniquely registered model version for this run"
    )
model_version = str(matching_versions[0].version)
print(
    json.dumps({"model_uri": model_info.model_uri, "version": model_version}, indent=2)
)

## Render and validate the approved serving deployment

The handoff below is complete, executable Python for the approved external platform process. Notebook execution itself remains read-only with respect to serving infrastructure. The final validator requires the endpoint to be ready and already routing 100% to the exact registered model version before invocation helpers are exposed.



In [ ]:
def served_entity(version: str) -> ServedEntityInput:
    environment_vars = {
        "DEEPAGENTS_MODEL_ENDPOINT": config.model_endpoint,
        "DEEPAGENTS_SQL_WAREHOUSE_ID": config.sql_warehouse_id,
        "DEEPAGENTS_DOCS_INDEX": config.docs_index,
        "DEEPAGENTS_SKILL_URI": config.skill_uri,
        "DEEPAGENTS_REQUIRE_REMOTE_SKILLS": "true",
        "DEEPAGENTS_APPLICATION": config.application,
        "DEEPAGENTS_ENVIRONMENT": config.environment,
        "DEEPAGENTS_RELEASE_VERSION": version,
        "MLFLOW_TRACKING_URI": "databricks",
        "MLFLOW_EXPERIMENT_ID": EXPERIMENT_ID,
        "ENABLE_MLFLOW_TRACING": "true",
    }
    if config.checkpointer_factory:
        environment_vars["DEEPAGENTS_CHECKPOINTER_FACTORY"] = (
            config.checkpointer_factory
        )
    return ServedEntityInput(
        name=f"deepagent-supervisor-{version}",
        entity_name=config.uc_model_name,
        entity_version=version,
        workload_size="Small",
        scale_to_zero_enabled=False,
        environment_vars=environment_vars,
    )


def endpoint_core(version: str) -> EndpointCoreConfigInput:
    entity = served_entity(version)
    return EndpointCoreConfigInput(
        name=config.serving_endpoint,
        served_entities=[entity],
        traffic_config=TrafficConfig(
            routes=[Route(served_entity_name=entity.name, traffic_percentage=100)]
        ),
    )


def platform_owned_tags() -> list[EndpointTag]:
    required = {
        "application": config.application,
        "project": config.project,
        "environment": config.environment,
        "team": os.environ.get("AAI_TEAM", ""),
        "owner_group": os.environ.get("AAI_OWNER_GROUP", ""),
        "cost_center": os.environ.get("AAI_COST_CENTER", ""),
        "data_classification": config.data_classification,
        "lifecycle": config.lifecycle,
        "repository": os.environ.get("AAI_REPOSITORY", ""),
        "release": os.environ.get("AAI_RELEASE", ""),
        "tag_schema_version": config.tag_schema_version,
    }
    missing = [key for key, value in required.items() if not value]
    if missing:
        raise RuntimeError(
            "Platform-controlled deployment tags are missing: "
            + ", ".join(sorted(missing))
        )
    return [EndpointTag(key=key, value=value) for key, value in required.items()]


def render_platform_owner_handoff(version: str) -> str:
    core_payload = endpoint_core(version).as_dict()
    tag_payload = [tag.as_dict() for tag in platform_owned_tags()]
    return f"""# Run only in the approved external platform process.
from datetime import timedelta
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import EndpointCoreConfigInput, EndpointTag

workspace = WorkspaceClient()
config = EndpointCoreConfigInput.from_dict({core_payload!r})
tags = [EndpointTag.from_dict(item) for item in {tag_payload!r}]
try:
    workspace.serving_endpoints.get({config.serving_endpoint!r})
except NotFound:
    workspace.serving_endpoints.create_and_wait(
        name={config.serving_endpoint!r},
        config=config,
        tags=tags,
        timeout=timedelta(minutes=30),
    )
else:
    workspace.serving_endpoints.update_config_and_wait(
        name={config.serving_endpoint!r},
        served_entities=config.served_entities,
        traffic_config=config.traffic_config,
        timeout=timedelta(minutes=30),
    )
"""


def require_deployed_endpoint(version: str) -> Any:
    workspace = WorkspaceClient()
    print(render_platform_owner_handoff(version))
    try:
        endpoint = workspace.serving_endpoints.get(config.serving_endpoint)
    except NotFound as exc:
        raise RuntimeError(
            "Serving endpoint is not provisioned; run the printed handoff through the platform process"
        ) from exc
    state = endpoint.state
    if (
        state is None
        or state.ready != EndpointStateReady.READY
        or state.config_update != EndpointStateConfigUpdate.NOT_UPDATING
    ):
        raise RuntimeError(f"Serving endpoint is not stable and ready: {state!r}")
    expected_name = f"deepagent-supervisor-{version}"
    entities = endpoint.config.served_entities if endpoint.config else []
    deployed = [
        entity
        for entity in entities or []
        if entity.name == expected_name
        and str(entity.entity_version) == version
        and entity.entity_name == config.uc_model_name
    ]
    routes = (
        endpoint.config.traffic_config.routes
        if endpoint.config and endpoint.config.traffic_config
        else []
    )
    exact_routes = [
        route
        for route in routes or []
        if route.served_entity_name == expected_name and route.traffic_percentage == 100
    ]
    if len(deployed) != 1 or len(exact_routes) != 1:
        raise RuntimeError(
            "The exact registered version is not receiving 100% traffic; run the printed handoff"
        )
    return endpoint


endpoint = require_deployed_endpoint(model_version)

## Invoke, resume HITL, and attach user feedback

The endpoint is a custom MLflow model, so requests use `dataframe_records`. New turns use an opaque UUIDv4 possession token; reuse the returned `thread_id` only when resuming its interrupt. With `InMemorySaver`, the resume is only reliable on the same process; a durable external checkpointer is required before scaling beyond a single replica.

Rating is a bounded tag. User comments are first-class MLflow feedback assessments—not tags—so user content never enters tag metadata.



In [ ]:
class Rating(StrEnum):
    THUMBS_UP = "thumbs_up"
    THUMBS_DOWN = "thumbs_down"


class FeedbackRequest(BaseModel):
    model_config = ConfigDict(extra="forbid", frozen=True, str_strip_whitespace=True)
    trace_id: str = Field(min_length=8, max_length=128)
    rating: Rating
    comment: str | None = Field(default=None, max_length=4_000)
    reviewer_group: str = Field(pattern=r"^group:[A-Za-z0-9_.:-]{1,120}$")


def _prediction(response: Any) -> dict[str, Any]:
    payload = response.as_dict() if hasattr(response, "as_dict") else dict(response)
    predictions = payload.get("predictions")
    if not isinstance(predictions, list) or len(predictions) != 1:
        raise RuntimeError(
            "Unexpected serving response; inspect the endpoint OpenAPI schema"
        )
    result = predictions[0]
    if not isinstance(result, dict):
        raise RuntimeError("Expected a structured agent prediction")
    return result


def invoke_agent(prompt: str) -> dict[str, Any]:
    if not prompt.strip() or len(prompt) > 50_000:
        raise ValueError("prompt must contain between 1 and 50,000 characters")
    thread_id = str(uuid4())
    record = {
        "thread_id": thread_id,
        "messages": [{"role": "user", "content": prompt}],
        "decisions": [],
    }
    response = WorkspaceClient().serving_endpoints.query(
        name=config.serving_endpoint,
        dataframe_records=[record],
    )
    return _prediction(response)


def resume_agent(
    thread_id: str,
    decisions: list[dict[str, Any]],
) -> dict[str, Any]:
    parsed = UUID(thread_id)
    if parsed.version != 4 or str(parsed) != thread_id.lower():
        raise ValueError(
            "thread_id must be the canonical UUIDv4 returned by invoke_agent"
        )
    record = {"thread_id": thread_id.lower(), "messages": [], "decisions": decisions}
    response = WorkspaceClient().serving_endpoints.query(
        name=config.serving_endpoint,
        dataframe_records=[record],
    )
    return _prediction(response)


def attach_user_feedback(request: FeedbackRequest) -> str:
    client = MlflowClient()
    client.set_trace_tag(request.trace_id, "user_feedback.rating", request.rating.value)
    client.set_trace_tag(
        request.trace_id,
        "user_feedback.has_comment",
        str(bool(request.comment)).lower(),
    )
    assessment = mlflow.log_feedback(
        trace_id=request.trace_id,
        name="user_satisfaction",
        value=request.rating == Rating.THUMBS_UP,
        rationale=request.comment,
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id=request.reviewer_group,
        ),
        metadata={"ui_rating": request.rating.value},
    )
    return str(assessment.assessment_id)


print("Helpers ready: invoke_agent(), resume_agent(), attach_user_feedback().")